# CartFlow (P06) — Week 6: Data Quality, Trusted Silver & Quarantine

**Working artifact:** `notebooks/04_data_quality_checks.ipynb`

This notebook implements the **Team 06 CartFlow Week-6 DQ method** using the approved CartFlow rule IDs and project table names. The worked reference is used only for the evaluation → explanation → routing → proof pattern; all project content below comes from CartFlow.

**Week-6 objective:** evaluate every Silver Candidate physical row, retain every applicable failure, route each physical row exactly once to Trusted Silver or Quarantine, and prove the split.

**Boundary:** Week 6 stops at DQ evaluation, Trusted/Quarantine, reconciliation and controlled replay. Gold/KPIs/Power BI/streaming are outside this notebook.

## Before running

The Week-6 conversion instructions require the Week-5 Candidate handoff, actual schema and baseline counts to be verified before DQ coding. The approved project rulebook is the authority for the eight DQ rules and their destinations.

The approved references also require:
- all eight exact CartFlow rule IDs;
- independent rule evaluation and multi-failure retention;
- entity-specific Trusted/Quarantine Delta outputs;
- Candidate = Trusted + Quarantine at distinct physical-record grain;
- zero Trusted/Quarantine overlap;
- independent item/payment aggregation with INR 0.05 reconciliation tolerance;
- controlled upstream/Candidate correction followed by a full applicable replay.

In [ ]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT current_catalog() AS active_catalog,
       current_schema() AS active_schema;

In [ ]:
%sql
SHOW TABLES LIKE 'silver_candidate_*';

**Checkpoint:** the five required inputs are `silver_candidate_orders`, `silver_candidate_order_items`, `silver_candidate_payments`, `silver_candidate_reviews`, and `silver_candidate_sellers`. If any is missing, stop and repair Week 5 before continuing.

In [ ]:
%sql
DESCRIBE TABLE silver_candidate_orders;

In [ ]:
%sql
DESCRIBE TABLE silver_candidate_order_items;

In [ ]:
%sql
DESCRIBE TABLE silver_candidate_payments;

In [ ]:
%sql
DESCRIBE TABLE silver_candidate_reviews;

In [ ]:
%sql
DESCRIBE TABLE silver_candidate_sellers;

In [ ]:
%sql
SELECT 'orders' AS entity, COUNT(*) AS candidate_rows, COUNT(DISTINCT source_record_id) AS distinct_physical_keys FROM silver_candidate_orders
UNION ALL SELECT 'order_items', COUNT(*), COUNT(DISTINCT source_record_id) FROM silver_candidate_order_items
UNION ALL SELECT 'payments', COUNT(*), COUNT(DISTINCT source_record_id) FROM silver_candidate_payments
UNION ALL SELECT 'reviews', COUNT(*), COUNT(DISTINCT source_record_id) FROM silver_candidate_reviews
UNION ALL SELECT 'sellers', COUNT(*), COUNT(DISTINCT source_record_id) FROM silver_candidate_sellers;

Record the actual baseline results from your Databricks run. Do not type copied example counts into the notebook.

# Part 2 — Approved CartFlow DQ contract

The eight governed rule IDs below are the exact CartFlow rules supplied in `ZENAIZ_Team06_CartFlow_Week06_DQ_Rules_v1.0`.

| Rule | Severity | Applies to | Fails when | Route |
|---|---|---|---|---|
| `DQ-ORD-001` | CRITICAL | Orders | `order_id` is null/blank/duplicated; `purchase_ts` is missing/unparseable; or `order_status` is missing/outside the approved lifecycle domain | `quarantine_orders` |
| `DQ-ITM-001` | CRITICAL | Order items | `order_item_id` missing/duplicated; `(order_id, order_item_seq)` not unique; or order/seller/product-category reference missing/unresolved | `quarantine_order_items` |
| `DQ-ORD-002` | CRITICAL | Orders | populated lifecycle timestamps are out of order, or status conflicts with required/allowed lifecycle timestamps | `quarantine_orders` |
| `DQ-MNY-001` | MAJOR | Items / payments | `item_price`, `freight_value` or `payment_value` is null where required, negative, or outside approved engineered limits | owning entity quarantine |
| `DQ-PAY-001` | CRITICAL | Payments | payment identity/reference/method/currency/installment sequence fails, or independent payment and item+freight totals differ by more than INR 0.05 | `quarantine_payments` |
| `DQ-SLR-001` | MAJOR | Sellers | seller identity/uniqueness/domain/effective window/item active-window check fails | `quarantine_sellers` |
| `DQ-REV-001` | MAJOR | Reviews | review identity/uniqueness/order eligibility/score/sentiment/date check fails | `quarantine_reviews` |
| `DQ-DLV-001` | CRITICAL | Orders | estimated/actual delivery dates or status agreement is invalid/incoherent or outside the approved project window | `quarantine_orders` |

**Approved dependency order:** Orders → Sellers → Order Items → Payments → Reviews.

**Fan-out guardrail:** aggregate items and payments independently to `order_id`, reconcile within INR 0.05, then join each summary once. Never raw-join items, payments and reviews to decide DQ or totals.

## Part 2.1 — Important rulebook gaps: do not invent values

The supplied CartFlow DQ rulebook says `DQ-MNY-001` uses **approved engineered limits**, and `DQ-DLV-001` refers to the **project window**, but the supplied rulebook/playbook pages do not provide numeric values for those limits/window boundaries.

Therefore this notebook **does not hard-code private values** such as `50000`, `5000`, `100000`, or arbitrary dates.

Enter mentor/approved values in the configuration view below before running the corresponding checks. Until they are supplied, the notebook can still inspect the non-negative/date-coherence portions, but final Week-6 acceptance for those rule components is **pending configuration**.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW cartflow_dq_config AS
SELECT
  CAST(NULL AS DECIMAL(18,2)) AS item_price_max,
  CAST(NULL AS DECIMAL(18,2)) AS freight_value_max,
  CAST(NULL AS DECIMAL(18,2)) AS payment_value_max,
  CAST(NULL AS DATE) AS project_window_start,
  CAST(NULL AS DATE) AS project_window_end,
  'DQ-CartFlow-W06-v1.0' AS ruleset_version;

In [ ]:
%sql
SELECT *,
  CASE WHEN item_price_max IS NULL OR freight_value_max IS NULL OR payment_value_max IS NULL
         THEN 'PENDING_APPROVED_MONEY_LIMITS'
       ELSE 'READY'
  END AS money_limit_status,
  CASE WHEN project_window_start IS NULL OR project_window_end IS NULL
         THEN 'PENDING_APPROVED_PROJECT_WINDOW'
       ELSE 'READY'
  END AS project_window_status
FROM cartflow_dq_config;

**Configuration gate:** replace only the `NULL` values in `cartflow_dq_config` with the values explicitly approved for CartFlow. Do not infer them from percentiles, observed maxima, another project.

# Part 3 — Evaluate and route `silver_candidate_orders`

Rules: `DQ-ORD-001`, `DQ-ORD-002`, `DQ-DLV-001`. Orders is a master entity and is evaluated first.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW orders_dup_check AS
SELECT order_id, COUNT(*) AS order_id_occurrences
FROM silver_candidate_orders
WHERE order_id IS NOT NULL AND trim(order_id) <> ''
GROUP BY order_id;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW orders_evaluated AS
SELECT
  o.*,
  COALESCE(d.order_id_occurrences, 0) AS order_id_occurrences,

  /* DQ-ORD-001: one visible result for the complete approved rule */
  CASE
    WHEN o.order_id IS NULL OR trim(o.order_id) = '' THEN 'FAIL'
    WHEN COALESCE(d.order_id_occurrences, 0) > 1 THEN 'FAIL'
    WHEN o.purchase_ts IS NULL THEN 'FAIL'
    WHEN o.order_status IS NULL OR trim(o.order_status) = ''
         OR o.order_status NOT IN ('shipped','delivered','cancelled','returned') THEN 'FAIL'
    ELSE 'PASS'
  END AS dq_ord001_check,

  /* DQ-ORD-002: adjacent populated lifecycle timestamps plus status alignment */
  CASE
    WHEN o.purchase_ts IS NOT NULL AND o.approval_ts IS NOT NULL AND o.approval_ts < o.purchase_ts THEN 'FAIL'
    WHEN o.approval_ts IS NOT NULL AND o.carrier_handoff_ts IS NOT NULL AND o.carrier_handoff_ts < o.approval_ts THEN 'FAIL'
    WHEN o.carrier_handoff_ts IS NOT NULL AND o.delivered_ts IS NOT NULL AND o.delivered_ts < o.carrier_handoff_ts THEN 'FAIL'
    WHEN o.delivered_ts IS NOT NULL AND o.return_ts IS NOT NULL AND o.return_ts < o.delivered_ts THEN 'FAIL'
    WHEN o.order_status = 'shipped' AND o.carrier_handoff_ts IS NULL THEN 'FAIL'
    WHEN o.order_status = 'shipped' AND o.delivered_ts IS NOT NULL THEN 'FAIL'
    WHEN o.order_status = 'delivered' AND o.delivered_ts IS NULL THEN 'FAIL'
    WHEN o.order_status = 'returned' AND (o.delivered_ts IS NULL OR o.return_ts IS NULL) THEN 'FAIL'
    WHEN o.order_status = 'cancelled' AND (o.delivered_ts IS NOT NULL OR o.return_ts IS NOT NULL) THEN 'FAIL'
    ELSE 'PASS'
  END AS dq_ord002_check,

  /* DQ-DLV-001: delivery/status coherence. Project-window bounds are configured separately. */
  CASE
    WHEN o.estimated_delivery_ts IS NULL THEN 'FAIL'
    WHEN o.purchase_ts IS NOT NULL AND o.estimated_delivery_ts < o.purchase_ts THEN 'FAIL'
    WHEN o.delivered_ts IS NOT NULL AND o.purchase_ts IS NOT NULL AND o.delivered_ts < o.purchase_ts THEN 'FAIL'
    WHEN o.return_ts IS NOT NULL AND o.purchase_ts IS NOT NULL AND o.return_ts < o.purchase_ts THEN 'FAIL'
    WHEN o.order_status = 'delivered' AND o.delivered_ts IS NULL THEN 'FAIL'
    WHEN o.order_status = 'returned' AND (o.delivered_ts IS NULL OR o.return_ts IS NULL) THEN 'FAIL'
    WHEN o.order_status = 'cancelled' AND (o.delivered_ts IS NOT NULL OR o.return_ts IS NOT NULL) THEN 'FAIL'
    WHEN o.order_status NOT IN ('delivered','returned') AND o.return_ts IS NOT NULL THEN 'FAIL'
    WHEN o.estimated_delivery_ts IS NOT NULL
         AND EXISTS (
           SELECT 1 FROM cartflow_dq_config c
           WHERE c.project_window_start IS NOT NULL
             AND c.project_window_end IS NOT NULL
             AND (
               to_date(o.estimated_delivery_ts) NOT BETWEEN c.project_window_start AND c.project_window_end
               OR (o.delivered_ts IS NOT NULL AND to_date(o.delivered_ts) NOT BETWEEN c.project_window_start AND c.project_window_end)
               OR (o.return_ts IS NOT NULL AND to_date(o.return_ts) NOT BETWEEN c.project_window_start AND c.project_window_end)
             )
         ) THEN 'FAIL'
    ELSE 'PASS'
  END AS dq_dlv001_check

FROM silver_candidate_orders o
LEFT JOIN orders_dup_check d ON o.order_id = d.order_id;

**Checkpoint:** every Candidate order still exists; the three rule columns only label the physical row.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW orders_explained AS
SELECT
  o.*,
  filter(array(
    CASE WHEN dq_ord001_check = 'FAIL' THEN 'DQ-ORD-001' END,
    CASE WHEN dq_ord002_check = 'FAIL' THEN 'DQ-ORD-002' END,
    CASE WHEN dq_dlv001_check = 'FAIL' THEN 'DQ-DLV-001' END
  ), x -> x IS NOT NULL) AS failed_rule_ids,
  filter(array(
    CASE WHEN dq_ord001_check = 'FAIL' THEN
      'order_id missing/blank/duplicated, purchase_ts missing/unparseable, or order_status outside approved lifecycle domain' END,
    CASE WHEN dq_ord002_check = 'FAIL' THEN
      'lifecycle timestamps or status-to-timestamp logic is inconsistent' END,
    CASE WHEN dq_dlv001_check = 'FAIL' THEN
      'delivery estimate/actual dates or delivery/status agreement is invalid or outside the configured project window' END
  ), x -> x IS NOT NULL) AS failure_reasons,
  filter(array(
    CASE WHEN dq_ord001_check = 'FAIL' THEN 'order_id, purchase_ts, order_status' END,
    CASE WHEN dq_ord002_check = 'FAIL' THEN 'purchase_ts, approval_ts, carrier_handoff_ts, delivered_ts, return_ts, order_status' END,
    CASE WHEN dq_dlv001_check = 'FAIL' THEN 'estimated_delivery_ts, delivered_ts, return_ts, order_status' END
  ), x -> x IS NOT NULL) AS affected_fields
FROM orders_evaluated;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW orders_final AS
SELECT
  e.*,
  CASE WHEN size(failed_rule_ids) = 0 THEN 'PASS' ELSE 'FAIL' END AS dq_status,
  CASE WHEN size(failed_rule_ids) = 0 THEN NULL ELSE 'CRITICAL' END AS severity,
  source_record_id AS physical_record_key,
  _source_file_name AS source_file,
  _ingestion_run_id AS batch_id,
  'DQ-CartFlow-W06-v1.0' AS dq_ruleset_version,
  current_timestamp() AS dq_checked_at,
  CASE WHEN size(failed_rule_ids) > 0 THEN current_timestamp() END AS quarantined_at,
  CASE WHEN size(failed_rule_ids) > 0 THEN 'pending_review' ELSE 'not_required' END AS rework_status
FROM orders_explained e;

In [ ]:
%sql
SELECT order_id, order_status, dq_status, failed_rule_ids, failure_reasons, affected_fields, severity
FROM orders_final
ORDER BY size(failed_rule_ids) DESC
LIMIT 20;

In [ ]:
%sql
CREATE OR REPLACE TABLE trusted_silver_orders USING DELTA AS
SELECT * FROM orders_final WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE quarantine_orders USING DELTA
AS SELECT * FROM orders_final WHERE dq_status = 'FAIL';

# Part 4 — Evaluate and route `silver_candidate_sellers`

Rule: `DQ-SLR-001` (MAJOR). Sellers is validated before child references.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW sellers_dup_check AS
SELECT seller_id, COUNT(*) AS seller_id_occurrences
FROM silver_candidate_sellers
WHERE seller_id IS NOT NULL AND trim(seller_id) <> ''
GROUP BY seller_id;

CREATE OR REPLACE TEMP VIEW seller_item_window_check AS
SELECT
  s.seller_id,
  MAX(CASE
        WHEN i.item_created_ts IS NOT NULL
             AND s.active_from IS NOT NULL
             AND s.active_to IS NOT NULL
             AND (to_date(i.item_created_ts) < s.active_from OR to_date(i.item_created_ts) > s.active_to)
        THEN 1 ELSE 0
      END) AS has_out_of_window_item
FROM silver_candidate_sellers s
LEFT JOIN silver_candidate_order_items i
  ON i.seller_id = s.seller_id
GROUP BY s.seller_id;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW sellers_evaluated AS
SELECT
  s.*,
  COALESCE(d.seller_id_occurrences,0) AS seller_id_occurrences,
  COALESCE(w.has_out_of_window_item,0) AS has_out_of_window_item,
  CASE
    WHEN s.seller_id IS NULL OR trim(s.seller_id) = '' THEN 'FAIL'
    WHEN COALESCE(d.seller_id_occurrences,0) > 1 THEN 'FAIL'
    WHEN s.seller_region IS NULL OR s.seller_region NOT IN ('South','West','North','East','Central') THEN 'FAIL'
    WHEN s.seller_type IS NULL OR s.seller_type NOT IN ('Individual','SME','Enterprise') THEN 'FAIL'
    WHEN s.service_band IS NULL OR s.service_band NOT IN ('Standard','Priority','Premium') THEN 'FAIL'
    WHEN s.seller_status IS NULL OR s.seller_status NOT IN ('active','inactive','suspended') THEN 'FAIL'
    WHEN s.active_from IS NULL OR s.active_to IS NULL OR s.active_to < s.active_from THEN 'FAIL'
    WHEN COALESCE(w.has_out_of_window_item,0) = 1 THEN 'FAIL'
    ELSE 'PASS'
  END AS dq_slr001_check
FROM silver_candidate_sellers s
LEFT JOIN sellers_dup_check d ON s.seller_id = d.seller_id
LEFT JOIN seller_item_window_check w ON s.seller_id = w.seller_id;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW sellers_final AS
SELECT
  e.*,
  CASE WHEN dq_slr001_check = 'FAIL' THEN 'FAIL' ELSE 'PASS' END AS dq_status,
  CASE WHEN dq_slr001_check = 'FAIL' THEN array('DQ-SLR-001') ELSE array() END AS failed_rule_ids,
  CASE WHEN dq_slr001_check = 'FAIL' THEN array('seller identity/domain/effective-date or item active-window check failed') ELSE array() END AS failure_reasons,
  CASE WHEN dq_slr001_check = 'FAIL' THEN array('seller_id, seller_region, seller_type, service_band, seller_status, active_from, active_to, item_created_ts') ELSE array() END AS affected_fields,
  source_record_id AS physical_record_key,
  _source_file_name AS source_file,
  _ingestion_run_id AS batch_id,
  'DQ-CartFlow-W06-v1.0' AS dq_ruleset_version,
  current_timestamp() AS dq_checked_at,
  CASE WHEN dq_slr001_check = 'FAIL' THEN 'MAJOR' END AS severity,
  CASE WHEN dq_slr001_check = 'FAIL' THEN current_timestamp() END AS quarantined_at,
  CASE WHEN dq_slr001_check = 'FAIL' THEN 'pending_review' ELSE 'not_required' END AS rework_status
FROM sellers_evaluated;

In [ ]:
%sql
SELECT seller_id, seller_region, seller_type, service_band, seller_status, dq_status, failed_rule_ids, failure_reasons
FROM sellers_final
ORDER BY dq_status DESC
LIMIT 20;

In [ ]:
%sql
CREATE OR REPLACE TABLE trusted_silver_sellers USING DELTA AS
SELECT * FROM sellers_final WHERE dq_status = 'PASS';

CREATE OR REPLACE TABLE quarantine_sellers USING DELTA
AS SELECT * FROM sellers_final WHERE dq_status = 'FAIL';

# Part 5 — Evaluate and route `silver_candidate_order_items`

Rules: `DQ-ITM-001` (CRITICAL) and `DQ-MNY-001` (MAJOR). References resolve only to **Trusted** Orders and Sellers.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW items_dup_id_check AS
SELECT order_item_id, COUNT(*) AS order_item_id_occurrences
FROM silver_candidate_order_items
WHERE order_item_id IS NOT NULL AND trim(order_item_id) <> ''
GROUP BY order_item_id;

CREATE OR REPLACE TEMP VIEW items_dup_seq_check AS
SELECT order_id, order_item_seq, COUNT(*) AS seq_occurrences
FROM silver_candidate_order_items
WHERE order_id IS NOT NULL AND trim(order_id) <> '' AND order_item_seq IS NOT NULL
GROUP BY order_id, order_item_seq;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW items_evaluated AS
SELECT
  i.*,
  COALESCE(id.order_item_id_occurrences,0) AS order_item_id_occurrences,
  COALESCE(sq.seq_occurrences,0) AS seq_occurrences,

  CASE
    WHEN i.order_item_id IS NULL OR trim(i.order_item_id) = '' THEN 'FAIL'
    WHEN COALESCE(id.order_item_id_occurrences,0) > 1 THEN 'FAIL'
    WHEN i.order_id IS NULL OR trim(i.order_id) = '' OR o.order_id IS NULL THEN 'FAIL'
    WHEN i.seller_id IS NULL OR trim(i.seller_id) = '' OR s.seller_id IS NULL THEN 'FAIL'
    WHEN i.order_id IS NULL OR i.order_item_seq IS NULL THEN 'FAIL'
    WHEN COALESCE(sq.seq_occurrences,0) > 1 THEN 'FAIL'
    WHEN i.product_id IS NULL OR trim(i.product_id) = '' THEN 'FAIL'
    WHEN i.category_code IS NULL OR trim(i.category_code) = '' THEN 'FAIL'
    ELSE 'PASS'
  END AS dq_itm001_check,

  CASE
    WHEN i.item_price IS NULL OR i.freight_value IS NULL THEN 'FAIL'
    WHEN i.item_price < 0 OR i.freight_value < 0 THEN 'FAIL'
    WHEN EXISTS (
      SELECT 1 FROM cartflow_dq_config c
      WHERE c.item_price_max IS NOT NULL AND i.item_price > c.item_price_max
    ) THEN 'FAIL'
    WHEN EXISTS (
      SELECT 1 FROM cartflow_dq_config c
      WHERE c.freight_value_max IS NOT NULL AND i.freight_value > c.freight_value_max
    ) THEN 'FAIL'
    ELSE 'PASS'
  END AS dq_mny001_check

FROM silver_candidate_order_items i
LEFT JOIN trusted_silver_orders o ON i.order_id = o.order_id
LEFT JOIN trusted_silver_sellers s ON i.seller_id = s.seller_id
LEFT JOIN items_dup_id_check id ON i.order_item_id = id.order_item_id
LEFT JOIN items_dup_seq_check sq
  ON i.order_id = sq.order_id AND i.order_item_seq = sq.order_item_seq;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW items_final AS
SELECT
  e.*,
  CASE WHEN dq_itm001_check='FAIL' OR dq_mny001_check='FAIL' THEN 'FAIL' ELSE 'PASS' END AS dq_status,
  filter(array(
    CASE WHEN dq_itm001_check='FAIL' THEN 'DQ-ITM-001' END,
    CASE WHEN dq_mny001_check='FAIL' THEN 'DQ-MNY-001' END
  ), x -> x IS NOT NULL) AS failed_rule_ids,
  filter(array(
    CASE WHEN dq_itm001_check='FAIL' THEN 'item/order/seller/product-category identity, uniqueness or reference resolution failed' END,
    CASE WHEN dq_mny001_check='FAIL' THEN 'item_price or freight_value is null/negative or outside the approved engineered limit' END
  ), x -> x IS NOT NULL) AS failure_reasons,
  filter(array(
    CASE WHEN dq_itm001_check='FAIL' THEN 'order_item_id, order_id, order_item_seq, seller_id, product_id, category_code' END,
    CASE WHEN dq_mny001_check='FAIL' THEN 'item_price, freight_value' END
  ), x -> x IS NOT NULL) AS affected_fields,
  source_record_id AS physical_record_key,
  _source_file_name AS source_file,
  _ingestion_run_id AS batch_id,
  'DQ-CartFlow-W06-v1.0' AS dq_ruleset_version,
  current_timestamp() AS dq_checked_at,
  CASE WHEN dq_itm001_check='FAIL' THEN 'CRITICAL'
       WHEN dq_mny001_check='FAIL' THEN 'MAJOR' END AS severity,
  CASE WHEN dq_itm001_check='FAIL' OR dq_mny001_check='FAIL' THEN current_timestamp() END AS quarantined_at,
  CASE WHEN dq_itm001_check='FAIL' OR dq_mny001_check='FAIL' THEN 'pending_review' ELSE 'not_required' END AS rework_status
FROM items_evaluated;

In [ ]:
%sql
SELECT order_item_id, order_id, seller_id, dq_status, failed_rule_ids, failure_reasons, severity
FROM items_final
ORDER BY size(failed_rule_ids) DESC
LIMIT 20;

In [ ]:
%sql
CREATE OR REPLACE TABLE trusted_silver_order_items USING DELTA AS
SELECT * FROM items_final WHERE dq_status='PASS';

CREATE OR REPLACE TABLE quarantine_order_items USING DELTA
AS SELECT * FROM items_final WHERE dq_status='FAIL';

# Part 6 — Evaluate and route `silver_candidate_payments`

Rules: `DQ-PAY-001` (CRITICAL) and `DQ-MNY-001` (MAJOR). Items and payments are aggregated independently before reconciliation.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW payment_id_check AS
SELECT payment_id, COUNT(*) AS payment_id_occurrences
FROM silver_candidate_payments
WHERE payment_id IS NOT NULL AND trim(payment_id) <> ''
GROUP BY payment_id;

CREATE OR REPLACE TEMP VIEW payment_installment_check AS
SELECT
  order_id,
  COUNT(*) AS installment_rows,
  COUNT(DISTINCT installment_no) AS distinct_installments,
  MIN(installment_no) AS min_installment,
  MAX(installment_no) AS max_installment
FROM silver_candidate_payments
WHERE order_id IS NOT NULL AND trim(order_id) <> ''
GROUP BY order_id;

CREATE OR REPLACE TEMP VIEW item_value_by_order AS
SELECT
  order_id,
  SUM(item_price * quantity) + SUM(freight_value) AS items_plus_freight_total
FROM trusted_silver_order_items
GROUP BY order_id;

CREATE OR REPLACE TEMP VIEW payment_value_by_order AS
SELECT order_id, SUM(payment_value) AS payments_total
FROM silver_candidate_payments
GROUP BY order_id;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW payments_evaluated AS
SELECT
  p.*,
  COALESCE(pi.payment_id_occurrences,0) AS payment_id_occurrences,
  ic.installment_rows, ic.distinct_installments, ic.min_installment, ic.max_installment,
  ivo.items_plus_freight_total,
  pvo.payments_total,

  CASE
    WHEN p.payment_id IS NULL OR trim(p.payment_id) = '' THEN 'FAIL'
    WHEN COALESCE(pi.payment_id_occurrences,0) > 1 THEN 'FAIL'
    WHEN p.order_id IS NULL OR trim(p.order_id) = '' OR o.order_id IS NULL THEN 'FAIL'
    WHEN p.payment_method IS NULL OR p.payment_method NOT IN ('card','upi','cod','wallet','net_banking') THEN 'FAIL'
    WHEN p.currency_code IS NULL OR p.currency_code <> 'INR' THEN 'FAIL'
    WHEN p.installment_no IS NULL THEN 'FAIL'
    WHEN ic.min_installment <> 1 OR ic.distinct_installments <> ic.installment_rows OR ic.max_installment <> ic.distinct_installments THEN 'FAIL'
    WHEN ivo.items_plus_freight_total IS NULL OR pvo.payments_total IS NULL THEN 'FAIL'
    WHEN ABS(pvo.payments_total - ivo.items_plus_freight_total) > 0.05 THEN 'FAIL'
    ELSE 'PASS'
  END AS dq_pay001_check,

  CASE
    WHEN p.payment_value IS NULL OR p.payment_value < 0 THEN 'FAIL'
    WHEN EXISTS (
      SELECT 1 FROM cartflow_dq_config c
      WHERE c.payment_value_max IS NOT NULL AND p.payment_value > c.payment_value_max
    ) THEN 'FAIL'
    ELSE 'PASS'
  END AS dq_mny001_check

FROM silver_candidate_payments p
LEFT JOIN trusted_silver_orders o ON p.order_id=o.order_id
LEFT JOIN payment_id_check pi ON p.payment_id=pi.payment_id
LEFT JOIN payment_installment_check ic ON p.order_id=ic.order_id
LEFT JOIN item_value_by_order ivo ON p.order_id=ivo.order_id
LEFT JOIN payment_value_by_order pvo ON p.order_id=pvo.order_id;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW payments_final AS
SELECT
  e.*,
  CASE WHEN dq_pay001_check='FAIL' OR dq_mny001_check='FAIL' THEN 'FAIL' ELSE 'PASS' END AS dq_status,
  filter(array(
    CASE WHEN dq_pay001_check='FAIL' THEN 'DQ-PAY-001' END,
    CASE WHEN dq_mny001_check='FAIL' THEN 'DQ-MNY-001' END
  ), x -> x IS NOT NULL) AS failed_rule_ids,
  filter(array(
    CASE WHEN dq_pay001_check='FAIL' THEN 'payment identity/reference/method/currency/installment or payment-to-item+freight reconciliation failed' END,
    CASE WHEN dq_mny001_check='FAIL' THEN 'payment_value is null/negative or outside the approved engineered limit' END
  ), x -> x IS NOT NULL) AS failure_reasons,
  filter(array(
    CASE WHEN dq_pay001_check='FAIL' THEN 'payment_id, order_id, payment_method, currency_code, installment_no, payment_value, order totals' END,
    CASE WHEN dq_mny001_check='FAIL' THEN 'payment_value' END
  ), x -> x IS NOT NULL) AS affected_fields,
  source_record_id AS physical_record_key,
  _source_file_name AS source_file,
  _ingestion_run_id AS batch_id,
  'DQ-CartFlow-W06-v1.0' AS dq_ruleset_version,
  current_timestamp() AS dq_checked_at,
  CASE WHEN dq_pay001_check='FAIL' THEN 'CRITICAL'
       WHEN dq_mny001_check='FAIL' THEN 'MAJOR' END AS severity,
  CASE WHEN dq_pay001_check='FAIL' OR dq_mny001_check='FAIL' THEN current_timestamp() END AS quarantined_at,
  CASE WHEN dq_pay001_check='FAIL' OR dq_mny001_check='FAIL' THEN 'pending_review' ELSE 'not_required' END AS rework_status
FROM payments_evaluated;

In [ ]:
%sql
SELECT payment_id, order_id, installment_no, payment_method, payment_value, dq_status, failed_rule_ids, failure_reasons, severity
FROM payments_final
ORDER BY size(failed_rule_ids) DESC
LIMIT 20;

In [ ]:
%sql
CREATE OR REPLACE TABLE trusted_silver_payments USING DELTA AS
SELECT * FROM payments_final WHERE dq_status='PASS';

CREATE OR REPLACE TABLE quarantine_payments USING DELTA
AS SELECT * FROM payments_final WHERE dq_status='FAIL';

# Part 7 — Evaluate and route `silver_candidate_reviews`

Rule: `DQ-REV-001` (MAJOR). Review parent eligibility is checked against Trusted Orders.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW review_id_check AS
SELECT review_id, COUNT(*) AS review_id_occurrences
FROM silver_candidate_reviews
WHERE review_id IS NOT NULL AND trim(review_id) <> ''
GROUP BY review_id;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW reviews_evaluated AS
SELECT
  r.*,
  COALESCE(ri.review_id_occurrences,0) AS review_id_occurrences,
  o.order_status AS parent_order_status,
  o.delivered_ts AS parent_delivered_ts,
  o.return_ts AS parent_return_ts,

  CASE
    WHEN r.review_id IS NULL OR trim(r.review_id) = '' THEN 'FAIL'
    WHEN COALESCE(ri.review_id_occurrences,0) > 1 THEN 'FAIL'
    WHEN r.order_id IS NULL OR trim(r.order_id) = '' OR o.order_id IS NULL THEN 'FAIL'
    WHEN o.order_status NOT IN ('delivered','returned') THEN 'FAIL'
    WHEN r.review_score IS NULL OR r.review_score NOT BETWEEN 1 AND 5 THEN 'FAIL'
    WHEN r.review_sentiment IS NULL OR r.review_sentiment NOT IN ('positive','neutral','negative') THEN 'FAIL'
    WHEN r.review_date IS NULL THEN 'FAIL'
    WHEN o.order_status='delivered' AND o.delivered_ts IS NOT NULL AND r.review_date < o.delivered_ts THEN 'FAIL'
    WHEN o.order_status='returned' AND o.return_ts IS NOT NULL AND r.review_date < o.return_ts THEN 'FAIL'
    ELSE 'PASS'
  END AS dq_rev001_check
FROM silver_candidate_reviews r
LEFT JOIN trusted_silver_orders o ON r.order_id=o.order_id
LEFT JOIN review_id_check ri ON r.review_id=ri.review_id;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW reviews_final AS
SELECT
  e.*,
  CASE WHEN dq_rev001_check='FAIL' THEN 'FAIL' ELSE 'PASS' END AS dq_status,
  CASE WHEN dq_rev001_check='FAIL' THEN array('DQ-REV-001') ELSE array() END AS failed_rule_ids,
  CASE WHEN dq_rev001_check='FAIL' THEN array('review identity/order eligibility/score/sentiment/date check failed') ELSE array() END AS failure_reasons,
  CASE WHEN dq_rev001_check='FAIL' THEN array('review_id, order_id, review_score, review_sentiment, review_date, parent order lifecycle') ELSE array() END AS affected_fields,
  source_record_id AS physical_record_key,
  _source_file_name AS source_file,
  _ingestion_run_id AS batch_id,
  'DQ-CartFlow-W06-v1.0' AS dq_ruleset_version,
  current_timestamp() AS dq_checked_at,
  CASE WHEN dq_rev001_check='FAIL' THEN 'MAJOR' END AS severity,
  CASE WHEN dq_rev001_check='FAIL' THEN current_timestamp() END AS quarantined_at,
  CASE WHEN dq_rev001_check='FAIL' THEN 'pending_review' ELSE 'not_required' END AS rework_status
FROM reviews_evaluated;

In [ ]:
%sql
SELECT review_id, order_id, review_score, review_sentiment, dq_status, failed_rule_ids, failure_reasons, severity
FROM reviews_final
ORDER BY dq_status DESC
LIMIT 20;

In [ ]:
%sql
CREATE OR REPLACE TABLE trusted_silver_reviews USING DELTA AS
SELECT * FROM reviews_final WHERE dq_status='PASS';

CREATE OR REPLACE TABLE quarantine_reviews USING DELTA
AS SELECT * FROM reviews_final WHERE dq_status='FAIL';

# Part 8 — Scorecard, reconciliation and membership proof

In [ ]:
%sql
WITH scorecard AS (
  SELECT 'DQ-ORD-001' AS rule_id, SUM(CASE WHEN dq_ord001_check='FAIL' THEN 1 ELSE 0 END) AS failed_rows FROM orders_final
  UNION ALL SELECT 'DQ-ORD-002', SUM(CASE WHEN dq_ord002_check='FAIL' THEN 1 ELSE 0 END) FROM orders_final
  UNION ALL SELECT 'DQ-DLV-001', SUM(CASE WHEN dq_dlv001_check='FAIL' THEN 1 ELSE 0 END) FROM orders_final
  UNION ALL SELECT 'DQ-SLR-001', SUM(CASE WHEN dq_slr001_check='FAIL' THEN 1 ELSE 0 END) FROM sellers_final
  UNION ALL SELECT 'DQ-ITM-001', SUM(CASE WHEN dq_itm001_check='FAIL' THEN 1 ELSE 0 END) FROM items_final
  UNION ALL SELECT 'DQ-MNY-001', SUM(CASE WHEN dq_mny001_check='FAIL' THEN 1 ELSE 0 END) FROM items_final
  UNION ALL SELECT 'DQ-PAY-001', SUM(CASE WHEN dq_pay001_check='FAIL' THEN 1 ELSE 0 END) FROM payments_final
  UNION ALL SELECT 'DQ-MNY-001', SUM(CASE WHEN dq_mny001_check='FAIL' THEN 1 ELSE 0 END) FROM payments_final
  UNION ALL SELECT 'DQ-REV-001', SUM(CASE WHEN dq_rev001_check='FAIL' THEN 1 ELSE 0 END) FROM reviews_final
)
SELECT rule_id, SUM(failed_rows) AS failed_physical_rows
FROM scorecard
GROUP BY rule_id
ORDER BY rule_id;

The scorecard is computed from one PASS/FAIL result per approved rule, so a multi-failure row contributes once to each failed rule and is never duplicated into multiple physical quarantine rows.

In [ ]:
%sql
SELECT 'orders' AS entity,
       COUNT(DISTINCT source_record_id) AS candidate_distinct,
       (SELECT COUNT(DISTINCT physical_record_key) FROM trusted_silver_orders) AS trusted_distinct,
       (SELECT COUNT(DISTINCT physical_record_key) FROM quarantine_orders) AS quarantine_distinct,
       COUNT(DISTINCT source_record_id)
         - (SELECT COUNT(DISTINCT physical_record_key) FROM trusted_silver_orders)
         - (SELECT COUNT(DISTINCT physical_record_key) FROM quarantine_orders) AS variance
FROM silver_candidate_orders
UNION ALL
SELECT 'order_items', COUNT(DISTINCT source_record_id),
       (SELECT COUNT(DISTINCT physical_record_key) FROM trusted_silver_order_items),
       (SELECT COUNT(DISTINCT physical_record_key) FROM quarantine_order_items),
       COUNT(DISTINCT source_record_id)
         - (SELECT COUNT(DISTINCT physical_record_key) FROM trusted_silver_order_items)
         - (SELECT COUNT(DISTINCT physical_record_key) FROM quarantine_order_items)
FROM silver_candidate_order_items
UNION ALL
SELECT 'sellers', COUNT(DISTINCT source_record_id),
       (SELECT COUNT(DISTINCT physical_record_key) FROM trusted_silver_sellers),
       (SELECT COUNT(DISTINCT physical_record_key) FROM quarantine_sellers),
       COUNT(DISTINCT source_record_id)
         - (SELECT COUNT(DISTINCT physical_record_key) FROM trusted_silver_sellers)
         - (SELECT COUNT(DISTINCT physical_record_key) FROM quarantine_sellers)
FROM silver_candidate_sellers
UNION ALL
SELECT 'payments', COUNT(DISTINCT source_record_id),
       (SELECT COUNT(DISTINCT physical_record_key) FROM trusted_silver_payments),
       (SELECT COUNT(DISTINCT physical_record_key) FROM quarantine_payments),
       COUNT(DISTINCT source_record_id)
         - (SELECT COUNT(DISTINCT physical_record_key) FROM trusted_silver_payments)
         - (SELECT COUNT(DISTINCT physical_record_key) FROM quarantine_payments)
FROM silver_candidate_payments
UNION ALL
SELECT 'reviews', COUNT(DISTINCT source_record_id),
       (SELECT COUNT(DISTINCT physical_record_key) FROM trusted_silver_reviews),
       (SELECT COUNT(DISTINCT physical_record_key) FROM quarantine_reviews),
       COUNT(DISTINCT source_record_id)
         - (SELECT COUNT(DISTINCT physical_record_key) FROM trusted_silver_reviews)
         - (SELECT COUNT(DISTINCT physical_record_key) FROM quarantine_reviews)
FROM silver_candidate_reviews;

**Acceptance condition:** variance must be 0 for all five entities.

In [ ]:
%sql
SELECT 'orders' AS entity,
  (SELECT COUNT(*) FROM trusted_silver_orders t JOIN quarantine_orders q ON t.physical_record_key=q.physical_record_key) AS intersecting_rows
UNION ALL SELECT 'order_items',
  (SELECT COUNT(*) FROM trusted_silver_order_items t JOIN quarantine_order_items q ON t.physical_record_key=q.physical_record_key)
UNION ALL SELECT 'sellers',
  (SELECT COUNT(*) FROM trusted_silver_sellers t JOIN quarantine_sellers q ON t.physical_record_key=q.physical_record_key)
UNION ALL SELECT 'payments',
  (SELECT COUNT(*) FROM trusted_silver_payments t JOIN quarantine_payments q ON t.physical_record_key=q.physical_record_key)
UNION ALL SELECT 'reviews',
  (SELECT COUNT(*) FROM trusted_silver_reviews t JOIN quarantine_reviews q ON t.physical_record_key=q.physical_record_key);

**Acceptance condition:** every `intersecting_rows` value must be 0.

In [ ]:
%sql
SELECT
  'orders' AS entity, COUNT(*) AS rows_with_multiple_failed_rules
FROM orders_final WHERE size(failed_rule_ids) > 1
UNION ALL SELECT 'order_items', COUNT(*) FROM items_final WHERE size(failed_rule_ids) > 1
UNION ALL SELECT 'payments', COUNT(*) FROM payments_final WHERE size(failed_rule_ids) > 1
UNION ALL SELECT 'reviews', COUNT(*) FROM reviews_final WHERE size(failed_rule_ids) > 1
UNION ALL SELECT 'sellers', COUNT(*) FROM sellers_final WHERE size(failed_rule_ids) > 1;

# Part 8.4 — Quarantine audit context and lineage

In [ ]:
%sql
SELECT order_id, physical_record_key, source_file, batch_id,
       failed_rule_ids, failure_reasons, affected_fields,
       severity, dq_checked_at, quarantined_at, rework_status
FROM quarantine_orders
LIMIT 20;

In [ ]:
%sql
SELECT 'orders' AS entity,
       SUM(CASE WHEN physical_record_key IS NULL OR source_file IS NULL OR batch_id IS NULL THEN 1 ELSE 0 END) AS missing_lineage
FROM quarantine_orders
UNION ALL SELECT 'order_items',
       SUM(CASE WHEN physical_record_key IS NULL OR source_file IS NULL OR batch_id IS NULL THEN 1 ELSE 0 END)
FROM quarantine_order_items
UNION ALL SELECT 'sellers',
       SUM(CASE WHEN physical_record_key IS NULL OR source_file IS NULL OR batch_id IS NULL THEN 1 ELSE 0 END)
FROM quarantine_sellers
UNION ALL SELECT 'payments',
       SUM(CASE WHEN physical_record_key IS NULL OR source_file IS NULL OR batch_id IS NULL THEN 1 ELSE 0 END)
FROM quarantine_payments
UNION ALL SELECT 'reviews',
       SUM(CASE WHEN physical_record_key IS NULL OR source_file IS NULL OR batch_id IS NULL THEN 1 ELSE 0 END)
FROM quarantine_reviews;

# Part 8.5 — Delta history evidence

In [ ]:
%sql
DESCRIBE HISTORY trusted_silver_orders;

In [ ]:
%sql
DESCRIBE HISTORY quarantine_orders;

In [ ]:
%sql
DESCRIBE HISTORY trusted_silver_sellers;

In [ ]:
%sql
DESCRIBE HISTORY quarantine_sellers;

In [ ]:
%sql
DESCRIBE HISTORY trusted_silver_order_items;

In [ ]:
%sql
DESCRIBE HISTORY quarantine_order_items;

In [ ]:
%sql
DESCRIBE HISTORY trusted_silver_payments;

In [ ]:
%sql
DESCRIBE HISTORY quarantine_payments;

In [ ]:
%sql
DESCRIBE HISTORY trusted_silver_reviews;

In [ ]:
%sql
DESCRIBE HISTORY quarantine_reviews;

# Part 9 — Controlled rerun / replay evidence

The conversion instructions require a controlled rerun with stable counts and a correction/replay demonstration. Never edit Quarantine directly.

First capture the current split. Then, after an approved upstream/Candidate correction, rerun the complete applicable DQ suite and compare the new route with the retained old evidence.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW week6_split_before AS
SELECT 'orders' AS entity, COUNT(*) AS candidate_rows,
       (SELECT COUNT(*) FROM trusted_silver_orders) AS trusted_rows,
       (SELECT COUNT(*) FROM quarantine_orders) AS quarantine_rows
FROM silver_candidate_orders
UNION ALL SELECT 'order_items', COUNT(*),
       (SELECT COUNT(*) FROM trusted_silver_order_items),
       (SELECT COUNT(*) FROM quarantine_order_items)
FROM silver_candidate_order_items
UNION ALL SELECT 'sellers', COUNT(*),
       (SELECT COUNT(*) FROM trusted_silver_sellers),
       (SELECT COUNT(*) FROM quarantine_sellers)
FROM silver_candidate_sellers
UNION ALL SELECT 'payments', COUNT(*),
       (SELECT COUNT(*) FROM trusted_silver_payments),
       (SELECT COUNT(*) FROM quarantine_payments)
FROM silver_candidate_payments
UNION ALL SELECT 'reviews', COUNT(*),
       (SELECT COUNT(*) FROM trusted_silver_reviews),
       (SELECT COUNT(*) FROM quarantine_reviews)
FROM silver_candidate_reviews;

SELECT * FROM week6_split_before ORDER BY entity;

## 9.1 Trace the required multi-failure order

Select an actual CartFlow order that fails both chronology and delivery rules. Do not type a copied example ID.

In [ ]:
%sql
SELECT order_id, physical_record_key, order_status,
       purchase_ts, approval_ts, carrier_handoff_ts, delivered_ts,
       estimated_delivery_ts, return_ts,
       failed_rule_ids, failure_reasons, affected_fields, severity
FROM quarantine_orders
WHERE array_contains(failed_rule_ids,'DQ-ORD-002')
  AND array_contains(failed_rule_ids,'DQ-DLV-001')
ORDER BY order_id
LIMIT 10;

## 9.2 Controlled correction template

1. Pick one actual quarantined `source_record_id`.
2. Identify the earliest incorrect upstream/Candidate value.
3. Correct that value through the governed source/Candidate rework process.
4. Do **not** update `quarantine_orders`.
5. Re-run the complete applicable DQ suite, not only the rule that previously failed.

If a Candidate-level rework is approved for demonstration, use a temporary rework view like the pattern below and replace only the selected record/value. The example intentionally contains placeholders rather than a fabricated order ID or correction.

In [ ]:
%sql
-- TEMPLATE ONLY: replace the placeholders after an approved correction is known.
-- Do not run with the placeholder strings.
--
-- CREATE OR REPLACE TEMP VIEW silver_candidate_orders_rework AS
-- SELECT
--   CASE WHEN source_record_id = '<ACTUAL_SOURCE_RECORD_ID>'
--        THEN '<CORRECTED_ORDER_ID_OR_VALUE>'
--        ELSE order_id END AS order_id,
--   ...
-- FROM silver_candidate_orders;
--
-- Then rerun the complete Orders evaluation (DQ-ORD-001, DQ-ORD-002, DQ-DLV-001)
-- against silver_candidate_orders_rework and rebuild Trusted/Quarantine from that
-- re-evaluated result. Retain the old quarantine evidence. Do not edit quarantine_orders.

# Part 10 — Controlled repeat-run stability check

In [ ]:
%sql
SELECT * FROM week6_split_before ORDER BY entity;

For a genuine rerun, execute the five evaluation/routing sections again against the unchanged Candidate inputs and capture a second split snapshot. The second snapshot should have the same candidate/trusted/quarantine counts, with no appended duplicate physical records. Keep the actual result as evidence.

In [ ]:
%sql
SELECT
  'orders' AS entity,
  COUNT(DISTINCT physical_record_key) AS trusted_distinct_after,
  (SELECT COUNT(DISTINCT physical_record_key) FROM quarantine_orders) AS quarantine_distinct_after
FROM trusted_silver_orders
UNION ALL SELECT 'order_items',
  COUNT(DISTINCT physical_record_key),
  (SELECT COUNT(DISTINCT physical_record_key) FROM quarantine_order_items)
FROM trusted_silver_order_items
UNION ALL SELECT 'sellers',
  COUNT(DISTINCT physical_record_key),
  (SELECT COUNT(DISTINCT physical_record_key) FROM quarantine_sellers)
FROM trusted_silver_sellers
UNION ALL SELECT 'payments',
  COUNT(DISTINCT physical_record_key),
  (SELECT COUNT(DISTINCT physical_record_key) FROM quarantine_payments)
FROM trusted_silver_payments
UNION ALL SELECT 'reviews',
  COUNT(DISTINCT physical_record_key),
  (SELECT COUNT(DISTINCT physical_record_key) FROM quarantine_reviews)
FROM trusted_silver_reviews;

# Part 11 — Week-6 acceptance checklist

Before submission, verify the real Databricks evidence shows:

- [ ] All eight exact CartFlow DQ rule IDs are implemented.
- [ ] Each approved rule is evaluated independently.
- [ ] All applicable failures remain on one physical row.
- [ ] `failed_rule_ids` has one occurrence of each failed rule ID; PASS rows have an empty array.
- [ ] Five Trusted/Quarantine entity pairs exist as Delta tables.
- [ ] Child references use Trusted Orders/Sellers in the approved dependency order.
- [ ] Seller active-window checks are executed.
- [ ] Items/payments are aggregated independently; reconciliation uses INR 0.05 tolerance.
- [ ] Candidate distinct physical records = Trusted distinct + Quarantine distinct for all five entities.
- [ ] Trusted/Quarantine intersection is zero for all five entities.
- [ ] Quarantine retains physical key, original Candidate fields, lineage, failure IDs/reasons, affected fields, severity and rework state.
- [ ] One actual multi-failure order is traced.
- [ ] One controlled correction is made upstream/in Candidate, then the full applicable suite is replayed.
- [ ] Old Quarantine evidence is retained; Quarantine is never manually edited.
- [ ] No Gold/KPI/Power BI/streaming implementation is added to Week 6.

## Week-6 repository/evidence boundary

Required repository working artifact: `notebooks/04_data_quality_checks.ipynb`.

The conversion guide also calls for:
- `docs/data_quality_summary.md` with the executed rule catalogue, real routing counts and impact;
- `screenshots/week06_*` with genuine rule/output/reconciliation/replay evidence;
- `weekly_logs/week06_log.md` with objectives, work, blockers, ownership and AI Transparency Note;
- `src/data_quality_rules.py` only if reusable Python/PySpark helpers are genuinely used.

Do not fabricate counts, PASS results, screenshots or Delta history. Week 6 ends at evaluated, explainable and reconciled Trusted Silver and Quarantine; Week 7 begins Gold modelling.